# ISMIR Analysis

In [1]:
from typing import Dict

In [2]:
from analyse_dataset import load_ismir_dataset, search_dataset
data = load_ismir_dataset(data_dir="dataset", years=[2021, 2022, 2023, 2024, 2025], verbose=True)

Loading ISMIR Markdown dataset from: 'dataset'...
  [Year 2021] Loaded 100 papers.
  [Year 2022] Loaded 113 papers.
  [Year 2023] Loaded 101 papers.
  [Year 2024] Loaded 123 papers.
  [Year 2025] Loaded 99 papers.
Total: 536 papers loaded across 5 year(s).



In [3]:
total_papers = sum(len(papers) for papers in data.values())
section_freq: Dict[str, int] = {}
for papers in data.values():
    for p in papers.values():
        for s in p["sections"].keys():
            section_freq[s] = section_freq.get(s, 0) + 1

print("Section Coverage Across Dataset:")
for sec, count in sorted(section_freq.items(), key=lambda x: x[1], reverse=True)[:12]:
    pct = count / total_papers * 100
    print(f"  {sec:<24} : {count:>4}/{total_papers} ({pct:>5.1f}%)")

Section Coverage Across Dataset:
  header                   :  536/536 (100.0%)
  references               :  532/536 ( 99.3%)
  abstract                 :  530/536 ( 98.9%)
  introduction             :  520/536 ( 97.0%)
  conclusion               :  313/536 ( 58.4%)
  related_work             :  249/536 ( 46.5%)
  experiments              :  152/536 ( 28.4%)
  acknowledgments          :  142/536 ( 26.5%)
  acknowledgements         :  140/536 ( 26.1%)
  results                  :  120/536 ( 22.4%)
  methodology              :   87/536 ( 16.2%)
  method                   :   81/536 ( 15.1%)


In [4]:
search_section = "abstract"
search = "ChatGPT"

target_secs = [search_section] if search_section else None
print(f"Searching for '{search}'" + (f" in section '{search_section}'" if search_section else "") + "...")
matches = search_dataset(data, search, target_sections=target_secs)
print(f"Found {len(matches)} match(es):\n")
for m in matches[:15]:
    print(f"  [{m['year']} - Paper {m['paper_id']}] Section: {m['section']}")
    print(f"    Title:   {m['title']}")
    print(f"    Snippet: {m['snippet']}\n")
if len(matches) > 15:
    print(f"  ... and {len(matches) - 15} more matches.")

Searching for 'ChatGPT' in section 'abstract'...
Found 1 match(es):

  [2024 - Paper 000078] Section: abstract
    Title:   Download pdf
    Snippet: ...lly we investigated whether ratings of music similarity, as measured via human listening tests, can be recovered from textual data by using ChatGPT. We examined the inferences that can be drawn from these experiments through the formal lens of validity. We showed that correlation of Cha...



---
# Location of AI-use disclosures (RQ2: location)

**Owner:** Hemanth. Logic in `disclosure_location.py`; `analyse_dataset.py` is unchanged.

**This section decides two things only:** whether a paragraph is an AI-use disclosure (the shared *gate*, rule in the
codebook sheet) and where it sits. Tasks, specificity and statement type are coded by teammates on the master list
exported at the end (`disclosures_master.xlsx`, keyed by `disclosure_id`), so nobody edits the same columns.

**Location codes:** `dedicated_statement` · `ethics_statement` · `acknowledgments` · `body_text` · `footnote` · `appendix`

**Steps:** A. coverage check → B. screen → C. export workbook → *code in Excel* → D. reliability → E. results → F. hand-off

In [5]:
# Run once if needed:
# %pip install pandas openpyxl matplotlib
import importlib
import pandas as pd
import disclosure_location as dl
importlib.reload(dl)          # picks up edits to disclosure_location.py without restarting
pd.set_option("display.max_colwidth", 160)
WORKBOOK = "location_coding.xlsx"

## A. Coverage check
Compares the dataset with the official paper counts (`EXPECTED_PAPERS` in the module; fill in the unknown years)
and lists paper numbers missing from each year's sequence. Report these gaps in the paper's method section.

In [6]:
pd.DataFrame(dl.coverage_report(data))

AttributeError: module 'disclosure_location' has no attribute 'coverage_report'

## B. Screen the dataset
One row per candidate paragraph:
* **named_tool**: a tool name anywhere in the paper (ChatGPT, GPT-4o, Claude 3.5, Copilot, DeepL, LLM, ...);
* **generic_wording**: no tool name, but generic AI wording (AI, language model, machine translation, writing
  assistant) plus a use or writing cue, inside acknowledgments / ethics / dedicated statement / footnote / appendix.

`priority` 3 = writing cue or non-use cue in a disclosure place · 2 = inside a disclosure place ·
1 = "we used / using <tool>" · 0 = plain mention. `auto_flags` = hints that a row is probably *not* a disclosure
(`link_only`, `funding_or_name`, `likely_third_party`); they never remove rows.

In [ ]:
hits = dl.screen_dataset(data)
hits_df = pd.DataFrame(hits)
print(f"{len(hits_df)} candidate paragraphs")
pd.DataFrame(dl.paper_counts(data, hits))

In [ ]:
# Candidates by year and priority, and by proposed location
display(pd.crosstab(hits_df.year, hits_df.priority, margins=True))
display(pd.crosstab(hits_df.year, hits_df.location_auto, margins=True))

In [ ]:
# Strongest candidates
cols = ["hit_id", "priority", "location_auto", "section_heading", "auto_flags", "text"]
hits_df.sort_values(["priority", "year"], ascending=[False, True])[cols].head(25)

## C. Export the coding workbook
| Sheet | Who codes it | What for |
|---|---|---|
| `to_code` | Hemanth | every priority ≥ 1 row |
| `recall_sample` | Hemanth | 50 random priority-0 rows → estimates what the screen misses |
| `reliability_10pct` | nobody directly (see C2) | ≥ 20 rows for Cohen's kappa; Hemanth's answers are pulled from the sheets above |
| `other_mentions` | nobody | remaining priority-0 rows, kept for transparency |
| `paper_counts`, `coverage`, `codebook` | — | denominators, data gaps, rules |

Orange columns: `is_disclosure` (Y/N/Unsure), `exclusion_reason` (when N), `location_verified` (only if
`location_auto` is wrong), `coder`, `notes`.

⚠️ Re-running this cell **overwrites** the workbook. After coding starts, do not run it again (or change `WORKBOOK`).

In [ ]:
dl.export_coding_workbook(hits, data, WORKBOOK)

### C2. File for the second coder
A separate file with only the reliability rows and empty coding columns, plus the codebook. Send it to one teammate;
they code it **without seeing your answers** and send it back. Takes about 15 minutes.

In [ ]:
SECOND_CODER_FILE = "reliability_second_coder.xlsx"
dl.export_second_coder_file(WORKBOOK, SECOND_CODER_FILE)

## D. Reliability (after you have coded `to_code` and the teammate has returned their file)
Cohen's kappa and % agreement on the gate (`is_disclosure`), and on location for rows both coders marked Y.

In [ ]:
SECOND_CODER_FILE = "reliability_second_coder.xlsx"   # the teammate's filled-in file
dl.reliability(WORKBOOK, second_coder_file=SECOND_CODER_FILE)

## E. Results (after coding)
* `by_paper`: papers per year with ≥ 1 disclosure in each location, papers disclosing, rate (RQ1/RQ2).
* `by_disclosure`: disclosure paragraphs per location (one paper can disclose in several places).
* `availability`: of the papers that **have** an ethics statement / acknowledgments, how many disclose there.
* `recall`: share of disclosures found in the random priority-0 sample (screening limitation).

In [ ]:
res = dl.summarise_verified(WORKBOOK, data)
display(res["progress"])
display(res["by_paper"])
display(res["by_disclosure"])
display(res["availability"])
display(res["recall"])
res["papers"]

In [ ]:
dl.plot_locations(res["by_paper"], "location_by_year.png");

## F. Hand-off to teammates
`disclosures_master.xlsx` lists every verified disclosure with its `disclosure_id`, location and text.
Tasks / specificity / statement-type coders copy it and add their own columns; they never edit `location_coding.xlsx`.

In [ ]:
dl.export_disclosure_list(WORKBOOK, "disclosures_master.xlsx")